# Session 12 · Homework Solutions (Teacher Copy) — The Accuracy Trap

**Machine Learning Foundations · Sanketana School of Code**

Worked solution with commentary. The ethics answers below **argue both sides** on purpose — a defensible verdict names the people who pay, and either verdict passes if the costs are correctly attributed.

**Acceptable variation:** either ethics verdict is fine when justified by stakeholders; the accuracy-is-misleading point must cite the baseline's 0.96 accuracy / 0.00 recall as evidence.

## Step 1 · The baseline that lies

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

fraud = pd.read_csv("../../../datasets/secondary/fraud_transactions.csv")
X = fraud.drop(columns=["is_fraud", "transaction_id"]).values
y = fraud["is_fraud"].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
bp = baseline.predict(X_test)
print("baseline accuracy:", round(accuracy_score(y_test, bp), 3))          # ~0.96
print("baseline recall:  ", round(recall_score(y_test, bp, zero_division=0), 3))  # 0.00

## Step 2 · A real model, judged properly

In [ ]:
model = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
pred = model.predict(X_test_s)
print("model accuracy :", round(accuracy_score(y_test, pred), 3))   # ~0.98
print("model precision:", round(precision_score(y_test, pred), 2))  # ~0.90
print("model recall   :", round(recall_score(y_test, pred), 2))     # ~0.60

## Step 3 · ✅ Why accuracy is the wrong headline

*"The always-legit baseline scores 0.96 accuracy but 0.00 recall — it catches none of the fraud, yet accuracy calls it excellent. Because fraud is only ~4% of transactions, accuracy is dominated by the easy legit majority and can't see whether the model does its actual job. I'd report **recall** to the fraud team, because their goal is catching fraud, and recall is exactly 'of all the real fraud, how much did we catch.'"*

## Step 4 · ✅ Ethics — both sides, then a stand

**Argue for 'false negatives are worse':** *"A missed fraud (false negative) means a thief succeeds — the customer loses money and the bank eats the loss and the reputational damage of failing to protect them. The cost falls on the defrauded customer and the bank. If the bank's mandate is to protect money, I'd call the miss worse and tune for higher recall."*

**Argue for 'false positives are worse':** *"A false alarm freezes an honest person's card — maybe at a petrol station at midnight, or paying for medicine. That customer bears a real, immediate, humiliating cost for something they didn't do, and the bank risks losing a loyal customer. If false alarms are frequent, thousands of innocent people pay for the bank's caution."*

**Either verdict passes** if it (a) names the specific mistake, (b) identifies **who bears the cost**, and (c) defends the choice. A bare preference with no stakeholders does not pass.

**Review talking point for Session 13:** all four outcomes — caught, missed, false alarm, fine — fit in one 2×2 grid, the **confusion matrix**. And the choice isn't all-or-nothing: by sliding the threshold we can catch *more* fraud (higher recall) at the price of *more* false alarms (lower precision). Next session we watch that trade-off move, and tie each setting to who it helps and who it harms.